# Artpedia Image Captioning — Full Pipeline

This notebook runs the complete pipeline for fine-tuning BLIP on Artpedia and evaluating the results:

1. **Setup** — GPU check, Drive mount, repo clone, dependency install
2. **Data prep** — download and cache Artpedia images to Google Drive
3. **Fine-tuning** — partially fine-tune BLIP's text decoder on the train split
4. **Evaluation** — BLEU-4 and CIDEr for pretrained vs fine-tuned
5. **t-SNE** — visualise how fine-tuning shifts decoder representations
6. **MLflow** — notes on viewing logged metrics

Images are cached to Google Drive so they survive runtime restarts.

## 1. Check GPU

Make sure the runtime is set to GPU: **Runtime → Change runtime type → T4 GPU** (or better).

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the Repo

Replace `USERNAME` with your GitHub username and `BRANCH_NAME` with the branch you want to run (e.g. `elpida`).

In [ ]:
# First-time clone — replace USERNAME and BRANCH_NAME before running.
!git clone -b BRANCH_NAME https://github.com/USERNAME/mscai-multimodal-project.git /content/mscai-multimodal-project
%cd /content/mscai-multimodal-project

# --- If the repo is already cloned, use this instead (comment out the clone above): ---
# %cd /content/mscai-multimodal-project
# !git pull

## 4. Install Dependencies

Installs LAVIS (editable from the bundled submodule) and all other requirements.
**A runtime restart is usually required after this cell** — Colab will prompt you, or use *Runtime → Restart session*. Re-run from cell 1 after restarting.

In [ ]:
!pip install -e ./LAVIS --quiet
!pip install -r requirements.txt --quiet

## 5. Configure Paths

Set `DATA_DIR` to the folder on your Drive where images and manifests are stored. Adjust the path to match your own Drive layout.

In [ ]:
# Adjust this path to wherever you store the data on your Google Drive.
DATA_DIR = "/content/drive/MyDrive/Github/mscai-multimodal-project/data"

print(f"DATA_DIR = {DATA_DIR}")

## 6. Data Preparation

Downloads Artpedia images from Wikimedia (500 px thumbnails) and writes manifests to Drive.
Images already on disk are reused — safe to rerun after interruptions.

Run the **test split first** (smaller, good for verifying setup). The train split cell is commented out — uncomment and run it deliberately once test is confirmed working.

Add `--rebuild-manifest` if you need to regenerate the manifest from scratch while reusing cached images (e.g. after a format change).

In [ ]:
# Download and cache the TEST split (326 images).
!python src/prepare_data.py \
    --split test \
    --output-dir "{DATA_DIR}" \
    --delay 2.0

In [ ]:
# Download and cache the TRAIN split (~2252 records; takes multiple sessions).
# Uncomment and run deliberately once test is confirmed working.

# !python src/prepare_data.py \
#     --split train \
#     --output-dir "{DATA_DIR}" \
#     --delay 2.0

# To regenerate the manifest from existing images (e.g. after a path format change):
# !python src/prepare_data.py --split train --output-dir "{DATA_DIR}" --rebuild-manifest

## 7. Fine-tuning

Fine-tunes BLIP's **text decoder** (last 2 transformer layers + cls head) while keeping the **vision encoder frozen**.

Checkpoints are saved to `{DATA_DIR}/../checkpoints/` after each epoch.
If you hit a CUDA OOM error, reduce `--batch-size` (try 4 or 2).

In [ ]:
!python src/train_caption.py \
    --manifest      "{DATA_DIR}/processed/train.jsonl" \
    --base-dir      "{DATA_DIR}" \
    --output-dir    "{DATA_DIR}/../checkpoints" \
    --epochs        5 \
    --batch-size    8 \
    --lr            1e-5 \
    --unfreeze-last-n 2 \
    --fp16 \
    --num-workers   2 \
    --use-mlflow \
    --mlflow-experiment blip-artpedia \
    --run-name      colab-run

## 8. Evaluation

Computes **BLEU-4** and **CIDEr** on the test split for both the pretrained base model and the fine-tuned checkpoint.
Results are saved to `eval_results.json` and optionally logged to MLflow.

Update `CHECKPOINT` to point to the epoch you want to evaluate.

In [ ]:
CHECKPOINT = f"{DATA_DIR}/../checkpoints/blip_artpedia_epoch5.pth"

!python src/evaluate_caption.py \
    --manifest      "{DATA_DIR}/processed/test.jsonl" \
    --base-dir      "{DATA_DIR}" \
    --checkpoint    "{CHECKPOINT}" \
    --split         test \
    --results-json  eval_results.json \
    --use-mlflow \
    --mlflow-experiment blip-artpedia \
    --run-name      colab-eval

## 9. t-SNE Comparison

Visualises how fine-tuning shifts the **decoder's multimodal representations** (cross-attention hidden states, not raw image embeddings) on the test set, coloured by painting century.

In [ ]:
!python src/tsne_compare.py \
    --manifest      "{DATA_DIR}/processed/test.jsonl" \
    --base-dir      "{DATA_DIR}" \
    --checkpoint    "{CHECKPOINT}" \
    --split         test \
    --output        tsne_compare.png

In [ ]:
# Display the saved t-SNE plot inline.
from IPython.display import Image
Image("tsne_compare.png")

## 10. MLflow — Viewing Results

MLflow writes logs to `./mlruns/` in the working directory (i.e. inside the cloned repo).

**To view the UI locally:**
1. Download the `mlruns/` folder from Colab (or from Drive if you redirected it).
2. On your local machine, from the repo root run:
   ```
   mlflow ui
   ```
   then open `http://127.0.0.1:5000` in your browser.

Alternatively, copy `mlruns/` to Drive to persist it across Colab sessions:
```python
import shutil
shutil.copytree('mlruns', f'{DATA_DIR}/../mlruns', dirs_exist_ok=True)
```